In [1]:
import datetime#Modül içe aktarılıyor. Tarih ve saat ile ilgili işlemleri kullanabilmek için.
import re#Regex (Tr:düzenli ifadeler) kütüphanesini içe aktarıyoruz. Metni kontrol etmek için.
from typing import Optional, List, Dict#fonksiyonlara ve değişkenlere type hinting(Tr:Tip ipuçları) eklemek için kullanılacak modüller içe aktarılıyor.

# Validation(Tr:veri doğrulama) helpers. Kütüphane sınıfı için veri doğrulama işlevlerini içeren yardımcı fonksiyonları tanımlıyoruz
_RE_TURKISH_TEXT = re.compile(r"^[A-Za-zÇĞİİÖŞÜçğıiöşü\s]+$")#re.Compile(...)(Tr:Derleme) verilen düzenli ifade desenini derler. Metnin sadece
#İngilizce-Türkçe harfler, boşluk karakteri (/s) içerip içermediğini kontrol eder. + ise bir veya daha fazla tekrarı temsil eder.

def validate_text(value: str, field: str) -> str:# metin doğrulama fonksiyonu oluşturuluyor.
    if not isinstance(value, str):#Tip kontrolü yapar. Gelen verinin sadece string(Tr:Metinsel ifade) olup olmadığını kontrol eder.
        raise ValueError(f"{field} metinsel olmalıdır.")#Şart sağlanıyorsa yani ifade string değilse hata fırlatır.
    v = value.strip()#.strip() boşlukları temizler ve değeri v değişkenine atar.
    if not v:#Metin boş ise exception(Tr:Hata)fırlatır.
        raise ValueError(f"{field} boş bırakılamaz.")
    if not _RE_TURKISH_TEXT.match(v):#Karakter kontrolü yapılıyor. Kitap adı veya yazar adı Regex desenine uymazsa hata fırlatır. Kontrol sağlayan şart yapısı.
        raise ValueError(f"{field} yalnızca harf ve boşluk içerebilir.")
    return v#Tüm kontrollerden geçtikten sonra,temizlemiş metin döndürülür.

def validate_positive_int(value, field: str) -> int:#Bu fonksiyon, bir değerin pozitif bir tam sayı olup olmadığını kontrol eder.
    if isinstance(value, str) and value.isdigit():#Eğer gelen değer  string ise ve sadece rakamlardan oluşuyorsa .isdigit()(built in gelen metot) onu tam sayıya dönüştürür.
        iv = int(value)
    elif isinstance(value, int):#Gelen değer zaten tam sayı ise doğrudan iv değişkenine atanır.
        iv = value
    else:
        raise ValueError(f"{field} pozitif bir tam sayı olmalıdır.")
    if iv <= 0:#Değerin negatif olmamasını kontrol eden şart yapısı.
        raise ValueError(f"{field} 1 veya daha büyük olmalıdır.")
    return iv

def validate_year(value) -> int:#Bu fonksiyon girilen basım yılının geçerli ve mantıksal sınırlar içinde olup olmadığını
# kontrol eder.->int çıktı tipi hakkında ipucu verir.Bu fonksiyon sununda tam sayı döndürecek.
    if isinstance(value, str) and value.isdigit():#value(giriş değeri) bir string mi diye kontrol eder. and iki koşulun da doğru olması gerektiğini belirtir.
    #.isdigit() value'nun içindeki karakterlerin hepsi rakam mı değil mi kontrolü yapar.
        y = int(value)#value tam sayıya dönüştürülür ve y değişkenine atanır.
    elif isinstance(value, int):# vslue zaten integer(Tr:tam sayı) ise doğrudan y değişkenine atanır.
        y = value
    else:
        raise ValueError("Basım yılı sayısal olmalıdır.")
    current = datetime.datetime.now().year#Sistemden o anki tarih ve saat bilgisini alır ve current(Tr:Mevcut) değişkenine atar
    if y < 1500 or y > current:#or en az biri veya ikisi de doğru ise çalışır. Girilen değer 1500 yılı ile mevcut yıl arasında olmalı.
        raise ValueError(f"Basım yılı 1500 ile {current} arasında olmalıdır.")
    return y

class Book:#Sınıf tanımlaması yapılır./Yeni bir kitap veri tipi oluşturulur.
    def __init__(self, kitap_adi: str, yazar: str, sayfa_sayisi, yil):#Constructor(Tr:Yapıcı metot) oluşturulur.Bu metot bir Book nesnesi/örneği/obje
#oluşturulduğunda otomatik olarak çaağrılır ve nesneye ilk değer ataması yapar.
#self:Nesnenin kendisini temsil eden zorunlu  bir parametredir.
        self.__kitap_adi = None#Niteliklere/Özellik/attribute başlangıç değeri( None(Tr:Boş)) atanır.
        #Özelliklerin başındaki underscore (__ Tr: Alt çizgi) dışarıdan doğrudan erişime kapalı olduğunu belirtir. Bu özelliklere getter-setter ile kontrollü erişim sağlanabiilir.
        self.__yazar = None
        self.__sayfa_sayisi = None
        self.__yil = None

        self.set_kitap_adi(kitap_adi)#setter çağrısı yapılır.Gelen kitap_adi değerini doğrudan atamakyerine,doğrulama ve biçimlendirme için set_kitap_adi metoduna gönderir.
        self.set_yazar(yazar)
        self.set_sayfa_sayisi(sayfa_sayisi)
        self.set_yil(yil)

        self.__borrowed = False#kitabın borrowed(Tr:Ödünçte) olup olmadığı bilgisini saklayacak bayrak değişkeni(boolean).
        self.__borrow_date: Optional[datetime.date] = None#Ödünç tarihini tutan özellik.Opsiyonel

    # Getter (Erişimci)/ Setter(Düzenleyici)
    def get_kitap_adi(self) -> str:#Kitabın adını okumak için kullanılır.
        return self.__kitap_adi

    def set_kitap_adi(self, value: str):#Kitabın adını dışarıdan değiştirmek için kullanılır
        self.__kitap_adi = validate_text(value, "Kitap adı").title()#Doğrulama ile atama yapılır. Yeni adı,validata_text fonksiyonuna göndererek doğrular

    def get_yazar(self) -> str:
        return self.__yazar

    def set_yazar(self, value: str):
        self.__yazar = validate_text(value, "Yazar").title()#.title() metni biçimlendiren built in(Python'un bize sunduğu kurulu) metot

    def get_sayfa_sayisi(self) -> int:
        return self.__sayfa_sayisi

    def set_sayfa_sayisi(self, value):
        self.__sayfa_sayisi = validate_positive_int(value, "Sayfa sayısı")#Gelen değeri,sadece pozitif bir tam sayı ise kabul eder.Doğrulama yapar.

    def get_yil(self) -> int:
        return self.__yil

    def set_yil(self, value):
        self.__yil = validate_year(value)#Gelen değeri validate_year fonksiyonuna gönderip istenen sayısal ve mantıksal aralıkta olması sağlanır.Sonucu soldaki değişkene atar.

    # Durum yönetimi
    def mark_borrowed(self):#Kitabın durumunu 'Ödünç alındı.' şeklinde güncelleyen metot
        if self.__borrowed:# Kitabın ödünçte olup olmadığını kontrol eder.
            raise RuntimeError("Bu kitap zaten ödünçtedir.")#Zaten ödünçteyse Hata fırlatır.
        self.__borrowed = True#if sağlanmıyorsa(kitap ödünçte değilse) durrumunu ödünçte olarqak günceller.
        self.__borrow_date = datetime.date.today()#fonksiyon ile güncel tarih alınır ve __borrow_date()(Tr:Ödünç tarihi) değişkenine atar.

    def mark_returned(self):#Kitabı geri alma işlemi yapan metot tanımlaması
        if not self.__borrowed:#Kitabın durumu kontrol edilir. Ödünçte değilse...
            raise RuntimeError("Bu kitap ödünç alınmamış.")#Hata fırlatır.
        self.__borrowed = False#Ödünç durumunu günceller. False: Ödünçte değil.
        self.__borrow_date = None#Ödünç alma tarihi boş olarak sıfırlanır.

    def is_borrowed(self) -> bool:#kitabın ödünç durmunu kontrol eden bayrak değişkeni.T/F
        return self.__borrowed

    # Polimorfizm için bilgi metodu
    def display_info(self) -> str:
        durum = "Ödünçte" if self.__borrowed else "Mevcut"#Kitabın durumu kontrol edilip durum değişkenine atanır
        return f"{self.__kitap_adi} — {self.__yazar} | {self.__sayfa_sayisi} s. | {self.__yil} | {durum}"

    def matches(self, kitap_adi: str, yazar: Optional[str] = None, yil: Optional[int] = None) -> bool: #arama kriteri ile nesneyi karşılaştırır. Eşleşiyor mu?
    #Optional:Opsiyonel parametre değer gönderilmezse varsayılan olarak None atanacak.
        if kitap_adi.strip().lower() != self.__kitap_adi.strip().lower():# Eğer aranan kitap_adi ile kitap nesnesinin adı eşleşmiyorsa...
        #ilk önce .strip() ile boşluklar temizlenir sonra .lower() ile tüm metin küçük harf yapılır ki büyük-küçük harf kriterine takılmasın
            return False
        if yazar is not None and yazar.strip().lower() != self.__yazar.strip().lower():#Yazar adı girilmişse ve uyuşmuyorsa
            return False
        if yil is not None and int(yil) != self.__yil:# Yıl girilmişse ve bu yıl kitap nesnesinin  yılıyla eşleşmiyorsa
            return False#False döndürür.Eşleşmiyor yani bulunamadı.
        return True#Yukarıdaki kontrollere takılmazsa uyuşur.

    def __str__(self) -> str:#Bu metot bir nesneyi print() ile yazdırırken ve string'e dönüştürürken Python tarafından otomatik olarak çağırılır.
        return self.display_info() #Nesnenin string temsilini oluşturmak için öncesinde tanımladığım display_info() metodu çağırılır.

# User (kapsüllemeli)
class User:#Kütüphane sistemindeki kullanıcıyı temsil eden sınıf tanımlaması yapıldı. Yeni bir "User" veri tipi oluşturuldu.
    def __init__(self, isim: str, numara: str, role: str = "student"):#Yapıcı metot:Bir User örneği(nesne) oluşturuldu otomatik olarak çağırılır ve nesneye ilk değerlerini verir.
    #role parametresine değer gönderilmezse varsayılan değeri "student" atanır
        self.__isim = None#Kullanıcının ismini tutacak private(Tr:Özel) alan.
        self.__numara = None
        self.__role = None

        self.set_isim(isim)#Gelen isim değerini, doğrulama yapması için set_isim metoduna gönderir.
        self.set_numara(numara)
        self.set_role(role)

        self.__borrowed_books: List[Book] = []#Kullanıcının ödünç aldığı Book nesnelerini saklayan boş liste oluşturulur. List[Book] listenin yalnız Book nesneleri içereceğini gösterir.

    # Getter / Setter
    def get_isim(self) -> str:#Kullanıcının adını okumak için kullanılır.
        return self.__isim#Saklanan adı döndürür

    def set_isim(self, value: str):#kullanıcı adını değiştirmek için kullanılır.
        self.__isim = validate_text(value, "Kullanıcı adı").title()#validate_text fonksiyonunu kullanarak doğrulama  yapılır ondan sonra __isim özelliğine değer atanır.

    def get_numara(self) -> str:
        return self.__numara

    def set_numara(self, value: str):
        if not isinstance(value, str) or not value.strip():#boş veya yanlış tip kontrolü yapılır.Value string değilse veya .strip() ile boşluklar atıldıktan sonra metin kalmıyorsa..
            raise ValueError("Numara boş bırakılamaz.")#Hata fırlatılır.
        self.__numara = value.strip()#Kontrol başarılıysa boşluklar atılır ve kalan ifade __numara değişkenine atanır.

    def get_role(self) -> str:
        return self.__role

    def set_role(self, value: str):
        self.__role = validate_text(value, "Rol")

    def borrow_book(self, book: Book):#Kitap ödünç alma işlemini gerçekleştiren metot
        if book in self.__borrowed_books:#Ödünçte mi kontrolü
            raise RuntimeError("Bu kullanıcı zaten aynı kitabı ödünç almış.")
        self.__borrowed_books.append(book)#Ödünçte değilse ödünçteki kitapların tutulduğu listeye eklenir.(.append() built in metodu ile)

    def return_book(self, book: Book):#Ödünçteki kitabın kütüphaneye iade edilmesini sağlayan metot
        if book not in self.__borrowed_books:#book'un ödünç alınan kitaplar listesinde olup olmadığını kontrol eder.
            raise RuntimeError("Kullanıcının ödünç kaydı bulunmuyor.")
        self.__borrowed_books.remove(book)#Ödünç listesinde .remove() medu ile silinir.

    def borrowed_list(self) -> List[str]:#Kullanıcının ödünç aldığı tüm kitapların adlarını içeren liste döndürmeye yarayan metot
        return [b.get_kitap_adi() for b in self.__borrowed_books]#Ödünç alınan he3r bir b kitabı için, kitabın adını alır ve bu adlardan yeni bir liste oluşturur.

    def has_borrowed(self, book: Book) -> bool:#Ödünç alınmış mı kontrolü yapan  metot
        return book in self.__borrowed_books

    def display_info(self) -> str:#Kullanıcı ismini, numarasını ve ödünç aldığı kitap sayısını(.len() ile) gösteren metot
        return f"{self.__isim} — No: {self.__numara} | Ödünçte: {len(self.__borrowed_books)}"

    def __str__(self) -> str:
        borrowed = ", ".join(self.borrowed_list()) or "Yok"
        return f"{self.display_info()} | Kitaplar: {borrowed}"

class AdminUser(User):# AdminUser User'dan kalıtım almaktadır.
    def __init__(self, isim: str, numara: str, role: str = "admin", yetki: str = "full", department: str = "library"):#Yapıcı metot User sınıfından gelen isim ve numara'ya ek, yöneticiye
    #özel yetki ve department parametrelerini ekledim. Parametrelere varsayılan değerler atadım.
        super().__init__(isim, numara, role)#super().__init___(...) miras alınan(User) ebeveyn sınıfın yapıcı metodunu çağırır. User sınıfınfan gelen özellikler User tarafından başlatılır.
        self.__yetki = yetki
        self.__department = validate_text(department, "Bölüm")#validate text fonksiyonu ile doğrulanıp öyle atanır.

    def get_yetki(self) -> str:
        return self.__yetki

    def set_yetki(self, value: str):
        self.__yetki = value

    def display_info(self) -> str:
        return f"[Admin] {self.get_isim()} — Yetki: {self.__yetki} | Dept: {self.__department}"
class Library:#Library sınıfı tanımlandı
    def __init__(self, name: str = "BALIKESİR Library", capacity: Optional[int] = 10):#Library nesnesi oluşturulduğunda çalıştırılacak
    #yapıcı metot. name ve capacity parametrelerini alır, varsayılan değerleri vardır.
        self.__name = validate_text(name, "Kütüphane adı")#Gelen name değerini metin doğrulama fonksiyonuna gönderir ve __name değişkenine atar.
        self.__kitaplar: List[Book] = []#Kütüphanedeki tüm Book nesnelerini tutacak boş bir liste oluşturur
        self.__users: Dict[str, User] = {}#Kütüphaneye kayıtlı tüm User nesnelerini tutacak boş bir sözlük yapısı oluşturduk.Arama kolaylığı sağlar.
        #Kullanıcı isimleri key(Tr:Anahtar) olarak kullanılır
        self.__capacity = capacity

    def get_name(self) -> str:
        return self.__name

    # Kullanıcı yönetimi
    def add_user(self, user: User) -> str:
        key = user.get_isim().strip().lower()#Kullanıcı adını alır, boşlukları siler ve tüm harfleri küçük hale getirir ardından key değişkenine atar.
        self.__users[key] = user#key ve User nesnesi sözlüğe eklenir.
        return "Kullanıcı eklendi."

    def create_or_get_user(self, isim: str, numara: str) -> User:#isim ve numara alarak bir user nesnesi döndüreceğini belirtir.
        key = isim.strip().lower()#kullanıcının isminden arama yapmak için anahtar oluşturur.
        if key in self.__users:#Bu anahtara sahip bir kullanıcı sözlükte var mı?
            return self.__users[key]#mevcut kullanıcıyı döndürür
        user = User(isim, numara)#Eğer kullanıcı bulunamazsa yeni bir user nesnesi oluşturur.
        self.__users[key] = user#Yeni oluşturulan kullanıcıyı sölüğe ekler.
        return user#Yeni oluşturulan user nesnesini döndürür.

    def get_user(self, isim: str) -> Optional[User]:#Kullanıcı alma metodu
        return self.__users.get(isim.strip().lower())#Gelen ismi temizler ve küçük harfe çevirir.self.__user.get(...) ile sözlük metodunu kullanarak kullanıcıyı arar.

    # Kitap yönetimi
    def add_book(self, kitap: Book) -> str:#kitap adında bir Book nesnesi alarak çalışan, string döndüren metot
        self.__kitaplar.append(kitap)#kitap nesnesini, self.__kitaplar kütüphanesine ekler.
        return "Kitap başarıyla kaydedildi."

    def remove_book(self, kitap_adi: str, yazar: str, yil) -> str:#kitabın adı,yazarı ve yılı bilgilerini alır ve string döndürür.
        try:#Hata yakalama başlangıcı
            kitap_adi = validate_text(kitap_adi, "Kitap adı")#Alınan verileri metin doğrulama fonksiyonunda kotnrol ettirip değişkene atar.
            yazar = validate_text(yazar, "Yazar")
            yil = validate_year(yil)
        except Exception as e:#try bloğunda hata oluşursa(e değişkenine atanır) bu blok çalışır
            return f"Girdi hatası: {e}"#Hatayı ekrana yazar

        for k in list(self.__kitaplar):#__kitaplar listesindeki tüm nesneleri dolaşacak döngü yapısı kurulur.
            if k.matches(kitap_adi, yazar, yil): #Eşleşme kontrolü yapılır.
                if k.is_borrowed():#Kitap ödünçteyse...
                    return "Kitap ödünçte olduğu için silinemez."
                self.__kitaplar.remove(k)#Ödünçte değilse döngünün üzerinde oldupu kitap silinir.
                return "Kitap kaydı silindi."
        return "Aranan kitap kayıtlarda bulunamadı."

    def list_books(self, filtre: Optional[Dict] = None) -> List[str]:#Kitap listeleme metodu. Stringlerden oluşan bir liste döndürür.
        results: List[str] = []#Filtreleme sonucunda listelenecek kitap bilgilerini tutacak boş liste oluşturulur
        for k in self.__kitaplar:#__kitaplar kütüphanesindeki her k nesnesi üzerinde dönecek döngü kuruldu.
            if not filtre:#filtre yoksa
                results.append(k.display_info())# döngünün üzerinde olan k nesnesinin bilgilerini result listesine eklenecek
                continue
            skip = False#kitabın filtreye takılıp takılmadıpını kontrol eder.
            if "yazar" in filtre and filtre["yazar"].strip().lower() != k.get_yazar().strip().lower():
                skip = True
            if "durum" in filtre:#filtre sözcüğünde durum anahtarı varsa çalışır
                durum = "oduncte" if k.is_borrowed() else "mevcut"
                if filtre["durum"].strip().lower() != durum:
                    skip = True
            if not skip:
                results.append(k.display_info())
        return results or ["Kayıtlı kitap bulunmuyor."]

    def find_book(self, kitap_adi: str) -> Optional[Book]:# İsteğe bağlı kitap nesnesi ya da None döneceğini belirtir.
        for k in self.__kitaplar:#Kütüphanedeki tüm kitaplar üzerinde tek tek dönecek.
            if k.get_kitap_adi().strip().lower() == kitap_adi.strip().lower():#Gelen kitap_adi ile nesnenin özellikleri uyuşuyorsa(Küçük-büyük harf eşleşmesi aramaz.)
                return k#k döndürülür
        return None

    # Ödünç alma
    def borrow(self, kitap_adi: str, yazar: str, yil, user_name: str, user_num: str) -> str:#Ödünç alacak kullanıcının adı,
    #numarası; kitabın ismi,yazarı,yılı bilgilerini alır. String ifade döndürür.
        try:#Hata yakalama başlatılır.
            kitap_adi = validate_text(kitap_adi, "Kitap adı")#Kullanıcı adı metin doğrulama fonsiyonuna gönderilir.Sonrasında değişkene atanır.
            yazar = validate_text(yazar, "Yazar")
            yil = validate_year(yil)
        except Exception as e:#hata oluştuğunda yakalanır ve hata mesajı ekrana yazdırlır.
            return f"Girdi hatası: {e}"

        kitap = next((k for k in self.__kitaplar if k.matches(kitap_adi, yazar, yil)), None)#kriterleri tamamen eşleşen kitap bulunur ve kitap değişkenine atanır bulunmazsa None atanır.
        if not kitap:#kitap bulunamadıysa
            return "İlgili kitap kayıtlarda bulunamadı."
        if kitap.is_borrowed():#kitap ödünçteyse
            return "Bu kitap şu anda ödünçte."

        user = self.create_or_get_user(user_name, user_num)#kullanıcı mevcutsa nesnesini alır değilse yeni bir kullanıcı oluşturulur ve nesneyi alır.
        try:
            kitap.mark_borrowed()#Kitap durumunu ödünçte olarak değştirir.
            user.borrow_book(kitap)#Kullanıcıya kitap eklenir.
            return f"{user.get_isim()} adlı okuyucu '{kitap.get_kitap_adi()}' kitabını ödünç aldı. İyi okumalar."
        except RuntimeError as e:
            return f"İşlem başarısız: {e}"

    def give_back(self, kitap_adi: str, yazar: str, yil, user_name: str) -> str:# Ödünçte kitabı iade alan metot
        try:
            kitap_adi = validate_text(kitap_adi, "Kitap adı")
            yazar = validate_text(yazar, "Yazar")
            yil = validate_year(yil)
        except Exception as e:
            return f"Girdi hatası: {e}"

        kitap = next((k for k in self.__kitaplar if k.matches(kitap_adi, yazar, yil)), None)#Kitap bulma __kitaplar listesinde döner ve
        #kriterlerin eşleşen ilk kitabı kitap değişkenine atar

        if not kitap:
            return "Kitap kayıtlarda bulunamadı."

        user = self.get_user(user_name)
        if not user:
            return "Belirtilen okuyucu sistemde kayıtlı değil."

        try:
            user.return_book(kitap)#Kitap kullanıcının listesinden kaldırılır.
            kitap.mark_returned()#Kitap durumunu iade edildi olarak günceller.
            return f"'{kitap.get_kitap_adi()}' başarıyla iade alındı. Teşekkür ederiz."
        except RuntimeError as e:
            return f"İşlem başarısız: {e}"

class LibraryApp:#Tüm sistemin başlatılmasını, yönetilmesini ve kullanıcı ile etkileşimini sağlayan sınıf tanımlanandı.
    def __init__(self, library: Optional[Library] = None):#Yapıcı metot. İsteğe bağlı olaqrak dışarıdan bir Library nesnesi alabilir.
        self.library = library if library is not None else Library("BALIKESİR Kütüphanesi")#Dışarıdan bir kütüphane listesi verilmezse varsayılan
        #BALIKESİR Kütüphanesi olan yeni Library nesnesi oluşturacak
        admin = AdminUser("Root Admin", "000", yetki="full", department="Library")#Yönetici nesnesi oluşturma.Admin sınıfında adı, numarası, tam yetkisi ve bölümmü belirtilen
        #bir yönetici nesnesi oluşturulur
        self.library.add_user(admin)#Oluşturulan admin nesnesi, kütüphane nesnesinin add_user metodu kullanılarak sisteme eklenir.

    def _prompt(self, msg: str) -> str:#Girdi alma metodu tanımlanıyor. msg adında bir string alır ve stringi temizleyip döndürür.
        return input(msg).strip()

    def run(self):#Ana çalıştırma metodu.
        print(f"{self.library.get_name()} — Hizmete hazırsınız. Hoş geldiniz.")#Kütüphane adını alır ve ekrana karşılama metni yazdırır.
        while True:#Sonsuz döngü başlatılıyor.
            print("\n1) Kitap Ekle  \n2) Kitap Sil  \n3) Kitapları Listele  \n4) Kullanıcı Oluştur/Getir \n5) Ödünç Ver \n6) Ödünç Geri Al \n7) Kullanıcı Bilgisi \n8) Çıkış")
            #Kullanıcıya seçim yapmasında yardımcı olarak seçenek menüsü yazdırılır.
            choice = self._prompt("Seçiminiz (1-8): ")#Kullanıcıdan yapmak istediği işlem alınır ve choice(Tr:Seçim) değişkenine atanır
            try:#Hata yakalama işlemi başlatılır
                if choice == "1":#Seçim 1 ise...
                    self._add_book_flow()#Metot çağırılır.
                elif choice == "2":#Seçim  ise...
                    self._remove_book_flow()#Metot çağırılır.
                elif choice == "3":#Seçim 3 ise...
                    self._list_books_flow()#Metot çağırılır.
                elif choice == "4":
                    self._create_user_flow()#Metot çağırılır.
                elif choice == "5":
                    self._borrow_flow()#Metot çağırılır.
                elif choice == "6":
                    self._return_flow()
                elif choice == "7":
                    self._user_info_flow()
                elif choice == "8":
                    print("Sistem kapatılıyor. İyi okumalar dilerim.")
                    break
                else:
                    print("Geçersiz seçim. Lütfen 1–8 arasında bir rakam girin.")
            except Exception as e:
                print(f"Beklenmeyen hata: {e}")

    # akış metotları
    def _add_book_flow(self):#Kitap ekleme metodu oluşturuldu.
        print("\n-- Kitap Ekleme --")
        kitap_adi = self._prompt("Kitap adı: ")#_prompt() metodu ile girdi alınır.
        yazar = self._prompt("Yazar: ")
        sayfa = self._prompt("Sayfa sayısı: ")
        yil = self._prompt("Basım yılı: ")
        try:
            kitap = Book(kitap_adi, yazar, sayfa, yil)#Book sınıfından nesne oluşturuldu.Book sınıfındaki set metotları veri doğrulaması yapar.
            res = self.library.add_book(kitap)#Oluşturulan kitap nesnesi, kütüphanenin add_book metoduna gönderilir.Sonuç res değişkenine atanır.
            print(res)#add_book metodu içindeki kitap eklenme mesajı ekrana yazdırılır.
        except Exception as e:
            print(f"Kitap eklenemedi: {e}")

    def _remove_book_flow(self):#kitap silme işlemini yapan metot tanımlandı.
        print("\n-- Kitap Silme --")
        kitap_adi = self._prompt("Kitap adı (tam): ")#Silinecek kitabın verileri alınır.
        yazar = self._prompt("Yazar (tam): ")
        yil = self._prompt("Basım yılı: ")
        res = self.library.remove_book(kitap_adi, yazar, yil)#Alınan verileri kütüphanenin remove_book metoduna gönderir ve bu metot silme işlemini gerçekleştir.
        print(res)

    def _list_books_flow(self):#Kitap listeleyen metot tanımlaması yapıldı.
        print("\n-- Kütüphane Kayıtları --")
        filtro = self._prompt("Filtrelemek ister misiniz? (yes/no): ").lower()
        filtre = None
        if filtro == "yes":
            yazar = self._prompt("Yazar (boş bırak = tümü): ")#Kullanıcıdan yazar adı girmesi istenir.
            durum = self._prompt("Durum filtre (mevcut/oduncte/boş=hepsi): ").lower()#Kullanıcı kitabın durumuna göre filtreleme yapmak istiyorsa
            filtre = {}#Filtre sözlüğü oluşturma.Kullanıcı filte yaqpmayı tercih ettiysefiltre değişkeni {}szölüğe dönüştürülür.
            if yazar:
                filtre["yazar"] = yazar
            if durum in ("mevcut", "oduncte"):
                filtre["durum"] = durum
        lines = self.library.list_books(filtre)
        for l in lines:
            print(" -", l)

    def _create_user_flow(self):#Kullanıcı oluşturma veya alma metodu
        print("\n-- Kullanıcı Oluştur --")
        isim = self._prompt("İsim: ")#_promt() metodu kullanılarak kullanıcının adı alınr.
        numara = self._prompt("Numara: ")
        try:
            user = self.library.create_or_get_user(isim, numara)# Alınan veriler ütüphanenin create_or_get_user() metoduna gönderilir. Bu metot
            #kullanıcının sözlükte olup olmadığını kontrol eder.
            print("Okuyucu hazır:", user)
        except Exception as e:
            print("Kullanıcı işlemi başarısız:", e)

    def _borrow_flow(self):#Kitap ödünç alma metodu.
        print("\n-- Ödünç Verme --")
        kitap_adi = self._prompt("Kitap adı (tam): ")
        yazar = self._prompt("Yazar (tam): ")
        yil = self._prompt("Basım yılı: ")
        isim = self._prompt("Okuyucu ismi: ")
        numara = self._prompt("Okuyucu numarası: ")
        res = self.library.borrow(kitap_adi, yazar, yil, isim, numara) #Toplanan kitap ve kullanıcı verilerini kütüphanenin borrow metoduna gönderir. borrow metodu
        #tüm doğrulama ve ödünç verme işlemlerini yapar ve bir metin döndürür.
        print(res)

    def _return_flow(self):#Kitabı geri alma metodu.
        print("\n-- Ödünç Geri Alma --")
        kitap_adi = self._prompt("Kitap adı (tam): ")
        yazar = self._prompt("Yazar (tam): ")
        yil = self._prompt("Basım yılı: ")
        isim = self._prompt("Okuyucu ismi: ")
        res = self.library.give_back(kitap_adi, yazar, yil, isim) #Toplanan kitap ve kullanıcının isim verilerini kütüphanenin give_back metoduna gönderir. give_back metodu
        #tüm doğrulama ve ödünç verme işlemlerini yapar ve bir metin döndürür. bu metin res değişkenine atanır.
        print(res)

    def _user_info_flow(self):#Kullanıcı bilgilerini gösteren metot
        isim = self._prompt("Okuyucu ismi: ")
        user = self.library.get_user(isim)#Kütüphanenin get_user metodu çağırılır.Eğer kullanıcı varsa User nesnesini yoksa None döndürür.
        if not user:#Kullanıcı bulunamadıysa
            print("Aradığınız okuyucu sistemde kayıtlı değil.")
            return
        print(user)#Kullanıcı bulunduysa, User nesnesini doprudan yazdırır. Bu işlem User sınıfındaki özel metot olan __str__ metodunun otomatik olarak çağırılması ile gerçekleşir.
# Çalıştırma bloğu
if __name__ == "__main__":
    app = LibraryApp()
    app.run()


BALIKESİR Kütüphanesi — Hizmete hazırsınız. Hoş geldiniz.

1) Kitap Ekle  
2) Kitap Sil  
3) Kitapları Listele  
4) Kullanıcı Oluştur/Getir 
5) Ödünç Ver 
6) Ödünç Geri Al 
7) Kullanıcı Bilgisi 
8) Çıkış
Seçiminiz (1-8): 1

-- Kitap Ekleme --
Kitap adı: Büşra
Yazar: Buse
Sayfa sayısı: 54
Basım yılı: 2000
Kitap başarıyla kaydedildi.

1) Kitap Ekle  
2) Kitap Sil  
3) Kitapları Listele  
4) Kullanıcı Oluştur/Getir 
5) Ödünç Ver 
6) Ödünç Geri Al 
7) Kullanıcı Bilgisi 
8) Çıkış
Seçiminiz (1-8): 4

-- Kullanıcı Oluştur --
İsim: büş
Numara: 1
Okuyucu hazır: Büş — No: 1 | Ödünçte: 0 | Kitaplar: Yok

1) Kitap Ekle  
2) Kitap Sil  
3) Kitapları Listele  
4) Kullanıcı Oluştur/Getir 
5) Ödünç Ver 
6) Ödünç Geri Al 
7) Kullanıcı Bilgisi 
8) Çıkış
Seçiminiz (1-8): 5

-- Ödünç Verme --
Kitap adı (tam): büşra
Yazar (tam): buse
Basım yılı: 2000
Okuyucu ismi: büş
Okuyucu numarası: 1
Büş adlı okuyucu 'Büşra' kitabını ödünç aldı. İyi okumalar.

1) Kitap Ekle  
2) Kitap Sil  
3) Kitapları Listele  
4) Ku